# Feature Engineering — Persistent Lyme Disease Study

**Purpose:** Build analysis-ready features from the pre-processed dataset.  
**Input:** `dataset_preprocessing_ready.csv`  
— all values are already numeric: `0`, `1`, `NaN`, or continuous.  
**Output:** `df_ready.pkl` (feature matrix ready for modelling)

**Steps**
1. Load & sanity-check input
2. Drop 100 % missing columns
3. Engineer symptom-domain features
4. Engineer co-infection features (ELISPOT)
5. Engineer Borrelia ELISPOT features
6. Engineer serology panel features
7. One-hot encode blood-test location
8. Antibiotic exposure features
9. Drop columns with > 70 % missing
10. Remove ID and zero-variance columns
11. Missingness summary

In [1]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)
print("Libraries loaded")

Libraries loaded


## 1. Load and sanity-check input

In [2]:
df = pd.read_csv("../dataset/dataset_preprocessing_ready.csv")
print("Shape:", df.shape)

# ── Sanity check: everything except Q49 must be numeric ──────────
non_num = [
    c for c in df.select_dtypes(exclude=[np.number]).columns
    if c != "Q49_blood_analysis_where"
]
assert len(non_num) == 0, f"Non-numeric columns found: {non_num}"
print("✓ All columns are numeric (Q49 kept as string for one-hot encoding)")

Shape: (301, 154)
✓ All columns are numeric (Q49 kept as string for one-hot encoding)


## 2. Drop fully-missing columns (100 % NaN)

In [3]:
missing_ratio = df.isna().mean()
cols_to_drop = missing_ratio[missing_ratio == 1.0].index.tolist()

if cols_to_drop:
    print(f"Dropping {len(cols_to_drop)} fully-missing columns: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)
else:
    print("No fully-missing columns found")

print("Shape after drop:", df.shape)

Dropping 2 fully-missing columns: ['Doxycycline_dose', 'Amoxicillin_dose']
Shape after drop: (301, 152)


## 3. Symptom-domain features

For each domain we create:
- `<domain>_symp` — 1 if any symptom in domain is present, 0 otherwise, NaN if all missing
- `num_<domain>_symp` — count of symptoms present (NaN if none were assessed)

The `NaN` guard prevents patients with no domain data from being
incorrectly coded as symptom-free.

In [5]:
def make_domain_features(df, domain_name, cols):
    """
    Create a binary 'any' flag and a symptom count for a symptom domain.
    Both are set to NaN for patients where all domain columns are missing.
    """
    present = [c for c in cols if c in df.columns]
    if not present:
        print(f"No columns found for domain '{domain_name}'")
        return df

    all_missing = df[present].isna().all(axis=1) # number of assessed items

    # Binary flag
    df[f"{domain_name}_symp"] = np.where(
        all_missing,
        np.nan,
        (df[present] == 1).any(axis=1).astype(int)
    )

    # Count — NaN when nothing was assessed
    df[f"num_{domain_name}_symp"] = df[present].sum(axis=1, min_count=1)

    print(f"  {domain_name}: {len(present)} source columns")
    return df


DOMAINS = {
    "skin": [
        "Q15 Bulls Eye", "Q16. Rash"
    ],
    "general_wellbeing": [
        "Q17. Sweats", "Q18 Sore throat", "Q19. Headac",
        "Q20. Swglands", "Q21. Sev Fat"
    ],
    "cardiac": [
        "Q24. Chest P.", "Q25. ShortB", "Q26. Palpit", "Q27. Lighthead."
    ],
    "rheumatological": [
        "Q28. JointP", "Q29.Moving", "Q30. Intensity",
        "Q31. JointSw", "Q32. MuscW.le", "Q33.MuscP"
    ],
    "neurological": [
        "Q35Facial", "Q36ArmHan", "Q37Numbn", "Q38Concen",
        "Q39Sleep", "Q40Vision", "Q41. Neck", "Q42Tinnit"
    ],
    "psychological": [
        "Q43Person", "Q44Mood", "Q46Anger", "Q47Anxiety"
    ],
}

print("Engineering symptom-domain features:")
for domain, cols in DOMAINS.items():
    df = make_domain_features(df, domain, cols)

# Total symptom count across all domains
domain_count_cols = [f"num_{d}_symp" for d in DOMAINS if f"num_{d}_symp" in df.columns]
df["num_total_symptoms"] = df[domain_count_cols].sum(axis=1, min_count=1)
print(f"\n  num_total_symptoms created from {len(domain_count_cols)} domain counts")

Engineering symptom-domain features:
  skin: 2 source columns
  general_wellbeing: 5 source columns
  cardiac: 4 source columns
  rheumatological: 6 source columns
  neurological: 8 source columns
  psychological: 4 source columns

  num_total_symptoms created from 6 domain counts


## 4. Co-infection features (ELISPOT)

In [6]:
COINF_COLS = [
    "Babesia", "Bartonella H", "Myco Pne", "Ehrl Ana", "Rickettsia",
    "Epstein B", "Chlamydia P", "Chlamyd trac", "Cytomigalo",
    "VZV IgG", "Anaplasma Phago", "Herpes Simplex", "Aspergillas",
    "Yersinia.1", "Candida"
]

coinf_present = [c for c in COINF_COLS if c in df.columns]

if coinf_present:
    all_missing = df[coinf_present].isna().all(axis=1)
    df["coinfections_elispot_any"] = np.where(
        all_missing, np.nan,
        (df[coinf_present] == 1).any(axis=1).astype(int)
    )
    df["coinfections_elispot_count"] = df[coinf_present].sum(axis=1, min_count=1)
    df.loc[all_missing, "coinfections_elispot_count"] = np.nan
    print(f"Co-infection ELISPOT: {len(coinf_present)} pathogens")
else:
    print("No co-infection ELISPOT columns found")

Co-infection ELISPOT: 15 pathogens


## 5. Borrelia ELISPOT features

In [7]:
BB_COLS = ["BB Full Anti", "BB Osp Mix", "BBLFA"]
bb_present = [c for c in BB_COLS if c in df.columns]

if bb_present:
    all_missing = df[bb_present].isna().all(axis=1)
    df["borrelia_elispot_any"] = np.where(
        all_missing, np.nan,
        (df[bb_present] == 1).any(axis=1).astype(int)
    )
    df["num_borrelia_elispot_pos"] = df[bb_present].sum(axis=1, min_count=1)
    df.loc[all_missing, "num_borrelia_elispot_pos"] = np.nan
    print(f"Borrelia ELISPOT: {len(bb_present)} antigens")
else:
    print("No Borrelia ELISPOT columns found")

Borrelia ELISPOT: 3 antigens


## 6. Serology panel features

Raw serology columns contain `P`/`N` codes (not yet recoded).  
Here we recode them to `1`/`0`, then compute:
- `IgG_total`, `IgM_total` — total positive IgG / IgM tests
- `num_positive_markers` — total positive across all serology
- `cluster_<pathogen>` — sum of positive IgG + IgM per organism

In [9]:
SERO_ALL = [
    "b.burg+afz+gar.IgG", "b.burg+afz+gar+IgM",
    "B.Burg Round Body IgG", "B.Burg Round body IgM",
    "BaB M IgG", "BaB M IgM",
    "Bart H IgG", "Bart H IgM",
    "Ehrl C IgG", "Ehrl IgM",
    "Rick Ak IgG", "Rick Ak IgM",
    "Coxs IgG", "Coxs IgM",
    "Epst B IgG", "Epst B IgM",
    "Hum Par IgG", "Hum Par IgM",
    "Mycop Pneu IgG", "Mycop Pneu IgM",
    "Chlamydia pneumoni",
    "HSV / IgG", "VZV/ IgG",
    "Toxoplasma", "Yersinia", "HSV/1gG"
]

sero_cols = [c for c in SERO_ALL if c in df.columns]

if sero_cols:
    # Recode P/N → 1/0 (columns may still be string from the CSV)
    for c in sero_cols:
        if df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].str.strip()
        df[c] = df[c].replace(["US", "NA", "", " "], np.nan)

    df[sero_cols] = df[sero_cols].replace({"P": 1, "N": 0, "p": 1, "n": 0})
    df[sero_cols] = df[sero_cols].apply(pd.to_numeric, errors="coerce")

    # Validate: only 0, 1, NaN allowed
    for c in sero_cols:
        valid = df[c].isin([0, 1]) | df[c].isna()
        unexpected = df.loc[~valid, c].unique()
        if len(unexpected) > 0:
            print(f"  {c}: unexpected values {unexpected} → set to NaN")
            df.loc[~valid, c] = np.nan

    # IgG / IgM totals
    igG_cols = [c for c in sero_cols if "igg" in c.lower()]
    igM_cols = [c for c in sero_cols if "igm" in c.lower()]
    if igG_cols:
        df["IgG_total"] = df[igG_cols].sum(axis=1, min_count=1)
    if igM_cols:
        df["IgM_total"] = df[igM_cols].sum(axis=1, min_count=1)

    df["sero_tested_count"]  = df[sero_cols].notna().sum(axis=1)
    df["num_positive_markers"] = df[sero_cols].sum(axis=1, min_count=1)

    # Pathogen clusters: sum of IgG + IgM per organism
    CLUSTERS = {
        "cluster_bburg":      ["burg"],
        "cluster_babesia":    ["bab"],
        "cluster_bartonella": ["bart"],
        "cluster_ehrlichia":  ["ehrl"],
        "cluster_rickettsia": ["rick"],
        "cluster_coxsackie":  ["coxs"],
        "cluster_ebv":        ["epst"],
        "cluster_parvo":      ["hum par"],
        "cluster_mycoplasma": ["mycop"],
        "cluster_chlamydia":  ["chlamydia"],
        "cluster_hsv":        ["hsv"],
        "cluster_vzv":        ["vzv"],
        "cluster_toxoplasma": ["toxoplasma"],
        "cluster_yersinia":   ["yersinia"],
    }

    for cluster_name, patterns in CLUSTERS.items():
        cc = [c for c in sero_cols if any(p in c.lower() for p in patterns)]
        if cc:
            df[cluster_name] = df[cc].sum(axis=1, min_count=1)

    print(f"  Serology features created from {len(sero_cols)} columns")
    print(f"  IgG columns ({len(igG_cols)}): {igG_cols}")
    print(f"  IgM columns ({len(igM_cols)}): {igM_cols}")
else:
    print("No serology columns found")

  Serology features created from 26 columns
  IgG columns (12): ['b.burg+afz+gar.IgG', 'B.Burg Round Body IgG', 'BaB M IgG', 'Bart H IgG', 'Ehrl C IgG', 'Rick Ak IgG', 'Coxs IgG', 'Epst B IgG', 'Hum Par IgG', 'Mycop Pneu IgG', 'HSV / IgG', 'VZV/ IgG']
  IgM columns (10): ['b.burg+afz+gar+IgM', 'B.Burg Round body IgM', 'BaB M IgM', 'Bart H IgM', 'Ehrl IgM', 'Rick Ak IgM', 'Coxs IgM', 'Epst B IgM', 'Hum Par IgM', 'Mycop Pneu IgM']


## 7. One-hot encode blood-test location

In [10]:
if "Q49_blood_analysis_where" in df.columns:
    df = pd.get_dummies(
        df,
        columns=["Q49_blood_analysis_where"],
        prefix="blood",
        dummy_na=False,
        dtype=int          # keep as integer, not float
    )
    new_cols = [c for c in df.columns if c.startswith("blood_")]
    print(f"Q49 one-hot encoded → {len(new_cols)} columns: {new_cols}")
else:
    print("Q49_blood_analysis_where not found — skipping one-hot encoding")

Q49 one-hot encoded → 3 columns: ['blood_G', 'blood_UK', 'blood_USA']


## 8. Antibiotic exposure features

In [11]:
ABX_COLS = [
    "Cefuroxime_dose", "Rifampicin_dose", "Lymecyclin_dose",
    "Azithromycin_dose", "Clarithromycin_dose", "Doxycycline_dose",
    "Amoxicillin_dose", "Valoid_dose", "Malarone_dose", "Diflucan_dose"
]
abx_cols = [c for c in ABX_COLS if c in df.columns]

if abx_cols:
    df[abx_cols] = df[abx_cols].apply(pd.to_numeric, errors="coerce")

    # Count antibiotics actually administered (dose > 0, not just recorded)
    df["num_antibiotics_administered"] = df[abx_cols].notna().sum(axis=1)

    print(f"Antibiotic features created from {len(abx_cols)} columns")
    print(f"Distribution of num_antibiotics_administered:")
    print(f"{df['num_antibiotics_administered'].value_counts().sort_index().to_dict()}")
else:
    print("No antibiotic dose columns found")

Antibiotic features created from 8 columns
Distribution of num_antibiotics_administered:
{0: 2, 1: 1, 2: 19, 3: 263, 4: 15, 5: 1}


## 9. Drop columns with > 70 % missing values

In [12]:
THRESHOLD = 0.70
missing_ratio = df.isna().mean()
cols_to_drop = missing_ratio[missing_ratio > THRESHOLD].index.tolist()

if cols_to_drop:
    print(f"Dropping {len(cols_to_drop)} columns with >{int(THRESHOLD*100)}% missing:")
    for c in sorted(cols_to_drop):
        print(f"  {c}: {missing_ratio[c]*100:.1f}%")
    df = df.drop(columns=cols_to_drop)
else:
    print(f"No columns exceed {int(THRESHOLD*100)}% missingness")

print(f"\nShape after dropping high-missingness columns: {df.shape}")

Dropping 35 columns with >70% missing:
  ANA: 81.1%
  Anaplasma Phago: 99.7%
  Aspergillas: 99.0%
  Azithromycin_dose: 82.7%
  Babesia: 94.0%
  Bartonella H: 92.0%
  Candida: 98.3%
  Chlamyd trac: 98.3%
  Chlamydia P: 84.1%
  Chlamydia pneumoni: 93.4%
  Clarithromycin_dose: 96.3%
  Cytomigalo: 94.0%
  Diflucan_dose: 98.0%
  Ehrl Ana: 89.7%
  Epstein B: 93.7%
  HSV / IgG: 98.7%
  HSV/1gG: 99.3%
  Herpes Simplex: 97.7%
  Malarone_dose: 97.7%
  Myco Pne: 91.4%
  Q8_Interval: 72.4%
  Rickettsia: 99.3%
  Toxoplasma: 99.3%
  VZV IgG: 98.3%
  VZV/ IgG: 99.0%
  Valoid_dose: 96.3%
  Yersinia: 99.7%
  Yersinia.1: 97.7%
  cluster_chlamydia: 93.4%
  cluster_hsv: 98.3%
  cluster_toxoplasma: 99.3%
  cluster_vzv: 99.0%
  cluster_yersinia: 99.7%
  coinfections_elispot_any: 72.8%
  coinfections_elispot_count: 72.8%

Shape after dropping high-missingness columns: (301, 155)


## 10. Remove ID and zero-variance columns

In [13]:
cols_to_remove = [c for c in ["ID"] if c in df.columns]

# Zero-variance (constant) columns
numeric_df = df.select_dtypes(include=[np.number])
constant_cols = [
    c for c in numeric_df.columns
    if numeric_df[c].std(skipna=True) == 0
]
if constant_cols:
    print(f"Constant columns found: {constant_cols}")
    cols_to_remove.extend(constant_cols)

cols_to_remove = list(set(cols_to_remove))
if cols_to_remove:
    df = df.drop(columns=cols_to_remove)
    print(f"Dropped {len(cols_to_remove)} columns: {cols_to_remove}")
else:
    print("No ID or constant columns to remove")

print(f"\nFinal shape: {df.shape}")

Constant columns found: ['persistent']
Dropped 2 columns: ['ID', 'persistent']

Final shape: (301, 153)


## 11. Missingness summary

In [14]:
missing_pct = df.isna().mean() * 100
medians     = df.median(numeric_only=True)

summary = pd.concat(
    [missing_pct.rename("missing_pct"), medians.rename("median")],
    axis=1
).sort_values("missing_pct", ascending=False)

median_missing = summary["missing_pct"].median()
print(f"Median missing percentage across all features: {median_missing:.1f}%")
print(f"Features with 0% missing: {(summary['missing_pct'] == 0).sum()}")
print(f"Features with >50% missing: {(summary['missing_pct'] > 50).sum()}")

summary

Median missing percentage across all features: 4.3%
Features with 0% missing: 6
Features with >50% missing: 31


,missing_pct,median
Melatonin_dose,64.784053,3.0
BBLFA,59.800664,0.0
BB Osp Mix,58.803987,1.0
BB Full Anti,57.807309,1.0
num_borrelia_elispot_pos,56.478405,1.0
...,...,...
gender,0.000000,1.0
blood_G,0.000000,1.0
blood_UK,0.000000,0.0
blood_USA,0.000000,0.0


## 12. Save

In [15]:
df.to_pickle("../dataset/df_ready.pkl")
df.to_csv("../dataset/df_ready.csv", index=False)

print(f"Saved df_ready.pkl and df_ready.csv")
print(f"Shape: {df.shape[0]} patients × {df.shape[1]} features")

Saved df_ready.pkl and df_ready.csv
Shape: 301 patients × 153 features
